In [2]:
# Same program as the .ipynb notebook, saved as a plain Python script.
# Run from the notebooks/ folder: python final.py
#
# | Section | What happens |
# |---|---|
# | 0 | Settings |
# | 1 | Data loading and sanity checks |
# | 2 | Base features |
# | 3 | Advanced features |
# | 4 | Adversarial validation (is test different from train?) |
# | 4b | Automatic tuning: feature count and LightGBM settings |
# | 5 | Models and cross-validation |
# | 6 | Time-based holdout check |
# | 7 | Ensemble |
# | 8 | Feature importance and files for the website |
# | 9 | Submission |

import os, glob, json, gc, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
LGB_SEEDS = [42, 7, 2026]      # LightGBM is averaged over several seeds
DATA = '../data'
CACHE = '../data/cache'        # feature cache, avoids recomputation on re-runs
ASSETS = '../app/assets'
SUBMIT = '../submission'
TEAM_ID = 'E86FDEB5'

USE_CACHE = True               # False = recompute features
DROP_AFTER_SIGNAL = False      # True = drop transactions after the signal date (see section 1)
ADV_DROP_THRESHOLD = 0.70      # if adversarial AUC is above this, drop the most shifted features
RUN_SEARCH = True              # section 4b: search for the best feature count and LightGBM settings
TOP_K_GRID = [30, 50, 80, 120, None]   # None = all features

for p in [CACHE, ASSETS, SUBMIT]:
    os.makedirs(p, exist_ok=True)
np.random.seed(SEED)
T0 = time.time()

# %% [markdown]
# ## 1. Data loading and sanity checks
# Read the five competition files and check them.
# 
# **What to look at:**
# - **Escalation rate** — share of training alerts that were escalated.
# - **Signal date ranges.** If test starts after train ends, we are predicting the future and section 6 matters most. If the ranges overlap, random cross-validation is appropriate.
# - **Share of transactions after the signal date.** Anything above zero means "future" information is present.

# %%
tr_sig = pd.read_csv(f'{DATA}/train_signals.csv')
te_sig = pd.read_csv(f'{DATA}/test_signals.csv')
tr_tx = pd.read_parquet(f'{DATA}/train_transactions.parquet')
te_tx = pd.read_parquet(f'{DATA}/test_transactions.parquet')
sample = pd.read_csv(sorted(glob.glob(f'{DATA}/sample_submission*.csv'))[0])

print('train signals', tr_sig.shape, '| test signals', te_sig.shape)
print('train tx', tr_tx.shape, '| test tx', te_tx.shape)
print('Escalation rate:', round(tr_sig['eskalatsiya'].mean(), 4))
for name, s in [('train', tr_sig), ('test', te_sig)]:
    d = pd.to_datetime(s['signal_sanasi'])
    print(f'Signal dates {name}: {d.min().date()} — {d.max().date()}')
assert set(sample.signal_id) == set(te_sig.signal_id), 'sample_submission does not match test_signals'

# numbers for the website
DATA_STATS = {
    'n_tx_train': int(len(tr_tx)), 'n_tx_test': int(len(te_tx)),
    'tx_per_alert_median': float(tr_tx.groupby('signal_id').size().median()),
    'train_dates': [str(pd.to_datetime(tr_sig.signal_sanasi).min().date()), str(pd.to_datetime(tr_sig.signal_sanasi).max().date())],
    'test_dates': [str(pd.to_datetime(te_sig.signal_sanasi).min().date()), str(pd.to_datetime(te_sig.signal_sanasi).max().date())],
}
print(DATA_STATS)

# %%
def signal_end(s):
    # a date without time is treated as the end of that day
    d = pd.to_datetime(s)
    if (d == d.dt.normalize()).all():
        d = d + pd.Timedelta(days=1)
    return d

def after_share(sig, tx):
    t = tx[['signal_id', 'tranzaksiya_vaqti']].merge(sig[['signal_id', 'signal_sanasi']], on='signal_id')
    return (pd.to_datetime(t['tranzaksiya_vaqti']) > signal_end(t['signal_sanasi'])).mean()

print('Share of transactions after the signal date: train', round(after_share(tr_sig, tr_tx), 4),
      '| test', round(after_share(te_sig, te_tx), 4))

# %% [markdown]
# ## 2. Base features
# One alert has dozens or hundreds of transactions, but the model needs one row per alert. We summarise each alert's history into numeric features:
# 
# - **Size statistics** of `miqdor_indeksi`: count, sum, mean, std, min, max, median, 90th percentile.
# - **Breakdowns** by direction (kirim = incoming, chiqim = outgoing), by type (karta, bank_otkazmasi, naqd, xalqaro) and by their combination: counts, sums, means, shares.
# - **Windows before the signal:** activity in the last 1, 3, 7, 14, 30 and 90 days.
# - **Recency and rhythm:** days since the last transaction, history length, gaps between transactions.
# - **Night and weekend** activity shares.
# - **Daily activity:** active days, max transactions per day, days with both incoming and outgoing flows.
# - **Ratios:** e.g. 7-day vs 30-day activity, which captures a burst right before the alert.
# 
# This cell only defines the functions; they are executed in section 3.

# %%
WINDOWS = [1, 3, 7, 14, 30, 90]

def flat(df, prefix):
    df.columns = [f'{prefix}_' + '_'.join(map(str, c)) if isinstance(c, tuple) else f'{prefix}_{c}'
                  for c in df.columns]
    return df

def build_features(sig, tx):
    tx = tx.merge(sig[['signal_id', 'signal_sanasi']], on='signal_id', how='left')
    tx['t'] = pd.to_datetime(tx['tranzaksiya_vaqti'])
    tx['dt'] = ((signal_end(tx['signal_sanasi']) - tx['t']).dt.total_seconds() / 86400).astype('float32')
    tx['amt'] = tx['miqdor_indeksi'].astype('float32')
    tx['dir'] = tx['kirim_chiqim'].astype(str)
    tx['typ'] = tx['tranzaksiya_turi'].astype(str)
    tx['dirtyp'] = (tx['dir'] + '_' + tx['typ']).astype('category')
    tx['dir'] = tx['dir'].astype('category'); tx['typ'] = tx['typ'].astype('category')
    tx = tx.sort_values(['signal_id', 't'])
    g = tx.groupby('signal_id')
    parts = []

    # 2.1 Overall size statistics
    base = g['amt'].agg(['count', 'sum', 'mean', 'std', 'min', 'max', 'median'])
    base['q90'] = g['amt'].quantile(0.9)
    total_cnt = base['count'].copy()
    parts.append(flat(base, 'all'))

    # 2.2 By direction, type and their combination
    for col in ['dir', 'typ', 'dirtyp']:
        p = tx.groupby(['signal_id', col], observed=True)['amt'].agg(['count', 'sum', 'mean', 'max']).unstack(col)
        p = flat(p, col)
        for c in [c for c in p.columns if c.startswith(f'{col}_count_')]:
            p[c.replace('_count_', '_share_')] = p[c] / total_cnt
        parts.append(p)

    # 2.3 Time windows before the signal
    for w in WINDOWS:
        m = (tx['dt'] >= 0) & (tx['dt'] <= w)
        parts.append(flat(tx[m].groupby('signal_id')['amt'].agg(['count', 'sum', 'max']), f'w{w}'))
        if w in (7, 30):
            a2 = tx[m].groupby(['signal_id', 'dir'], observed=True)['amt'].agg(['count', 'sum']).unstack('dir')
            parts.append(flat(a2, f'w{w}'))

    # 2.4 Recency, history length, gaps between transactions
    rec = g['dt'].agg(['min', 'max'])
    rec.columns = ['days_since_last', 'history_days']
    tx['after'] = (tx['dt'] < 0).astype('int8')
    rec['n_after_signal'] = tx.groupby('signal_id')['after'].sum()
    tx['gap_h'] = g['t'].diff().dt.total_seconds() / 3600
    gap = tx.groupby('signal_id')['gap_h'].agg(['mean', 'median', 'min', 'std'])
    parts += [rec, flat(gap, 'gap')]

    # 2.5 Night and weekend activity
    tx['night'] = tx['t'].dt.hour.isin([0, 1, 2, 3, 4, 5]).astype('int8')
    tx['weekend'] = (tx['t'].dt.dayofweek >= 5).astype('int8')
    parts.append(tx.groupby('signal_id')[['night', 'weekend']].mean().add_suffix('_share'))

    # 2.6 Daily activity, days with both incoming and outgoing flows
    tx['day'] = tx['t'].dt.floor('D')
    daily = tx.groupby(['signal_id', 'day', 'dir'], observed=True).size().unstack('dir', fill_value=0)
    dfeat = pd.DataFrame(index=daily.index.get_level_values(0).unique())
    dcnt = daily.sum(axis=1).groupby(level=0)
    dfeat['active_days'] = dcnt.size()
    dfeat['max_tx_per_day'] = dcnt.max()
    dfeat['std_tx_per_day'] = dcnt.std()
    if {'kirim', 'chiqim'} <= set(daily.columns):
        both = ((daily['kirim'] > 0) & (daily['chiqim'] > 0)).groupby(level=0)
        dfeat['days_in_and_out'] = both.sum()
        dfeat['share_days_in_and_out'] = both.mean()
    parts.append(dfeat)

    X = sig[['signal_id']].set_index('signal_id').join(pd.concat(parts, axis=1))

    # 2.7 Ratios: burst before the signal, flow balance
    eps = 1e-6
    X['ratio_w7_w30_cnt'] = X['w7_count'] / (X['w30_count'] + eps)
    X['ratio_w30_w90_cnt'] = X['w30_count'] / (X['w90_count'] + eps)
    X['ratio_w30_all_cnt'] = X['w30_count'] / (X['all_count'] + eps)
    X['tx_per_active_day'] = X['all_count'] / (X['active_days'] + eps)
    if 'dir_sum_kirim' in X and 'dir_sum_chiqim' in X:
        X['net_flow'] = X['dir_sum_kirim'].fillna(0) - X['dir_sum_chiqim'].fillna(0)
        X['in_out_cnt_ratio'] = X['dir_count_kirim'] / (X['dir_count_chiqim'] + eps)

    # 2.8 Signal date feature
    X['signal_dow'] = pd.to_datetime(sig.set_index('signal_id')['signal_sanasi']).dt.dayofweek

    # Counters for alerts without transactions = 0
    cnt_like = [c for c in X.columns if '_count' in c or c.startswith(('active_days', 'days_in', 'n_after'))]
    X[cnt_like] = X[cnt_like].fillna(0)
    return X.astype('float32')

# %% [markdown]
# ## 3. Advanced features
# Features that describe typical money-laundering patterns:
# 
# - **Time-decayed activity:** each transaction is weighted by `exp(-days/τ)` — the closer to the alert, the higher the weight.
# - **Pass-through flows:** money arrives and leaves shortly afterwards, often for a similar amount.
# - **Repeated amounts:** many transactions of the same size (possible structuring).
# - **Last 10 transactions:** their size and share of cash and international transfers.
# - **Type diversity:** entropy of the transaction-type distribution.
# - **Unusual recent amounts:** how much larger the last week's amounts are compared with the alert's own history.
# 
# The second cell computes all features for train and test. **This is the slowest step** on the first run; later runs load them from the cache.

# %%
def build_extra(sig, tx):
    tx = tx.merge(sig[['signal_id', 'signal_sanasi']], on='signal_id', how='left')
    tx['t'] = pd.to_datetime(tx['tranzaksiya_vaqti'])
    tx['dt'] = ((signal_end(tx['signal_sanasi']) - tx['t']).dt.total_seconds() / 86400).astype('float32')
    tx['amt'] = tx['miqdor_indeksi'].astype('float32')
    tx['is_in'] = (tx['kirim_chiqim'] == 'kirim').astype('int8')
    tx['is_naqd'] = (tx['tranzaksiya_turi'] == 'naqd').astype('int8')
    tx['is_xalq'] = (tx['tranzaksiya_turi'] == 'xalqaro').astype('int8')
    tx = tx.sort_values(['signal_id', 't']).reset_index(drop=True)
    F = pd.DataFrame(index=sig['signal_id'])
    dpos = tx['dt'].clip(lower=0)

    # 3.1 Time-decayed activity
    for tau in (3, 14, 60):
        w = np.exp(-dpos / tau).astype('float32')
        tx['_w'] = w; tx['_wa'] = w * tx['amt']
        tx['_wi'] = w * tx['is_in']; tx['_wn'] = w * tx['is_naqd']; tx['_wx'] = w * tx['is_xalq']
        a = tx.groupby('signal_id')[['_w', '_wa', '_wi', '_wn', '_wx']].sum()
        F[f'decay{tau}_cnt'] = a['_w']
        F[f'decay{tau}_amt'] = a['_wa']
        F[f'decay{tau}_in_share'] = a['_wi'] / (a['_w'] + 1e-6)
        F[f'decay{tau}_naqd_share'] = a['_wn'] / (a['_w'] + 1e-6)
        F[f'decay{tau}_xalq_share'] = a['_wx'] / (a['_w'] + 1e-6)
    F['decay_ratio_3_60'] = F['decay3_cnt'] / (F['decay60_cnt'] + 1e-6)

    # 3.2 Pass-through: outgoing transaction shortly after the latest incoming one
    tx['_kt'] = tx['t'].where(tx['is_in'] == 1)
    tx['_ka'] = tx['amt'].where(tx['is_in'] == 1)
    g = tx.groupby('signal_id')
    tx['_kt'] = g['_kt'].ffill(); tx['_ka'] = g['_ka'].ffill()
    out = tx[tx['is_in'] == 0].copy()
    out['h_after_in'] = (out['t'] - out['_kt']).dt.total_seconds() / 3600
    out['amt_diff'] = (out['amt'] - out['_ka']).abs()
    for h in (1, 24, 72):
        out[f'_q{h}'] = (out['h_after_in'] <= h).astype('int8')
    out['_pass'] = ((out['h_after_in'] <= 72) & (out['amt_diff'] < 0.1)).astype('int8')
    go = out.groupby('signal_id')
    for h in (1, 24, 72):
        F[f'out_within_{h}h_cnt'] = go[f'_q{h}'].sum()
        F[f'out_within_{h}h_share'] = go[f'_q{h}'].mean()
    F['passthrough_cnt'] = go['_pass'].sum()
    F['passthrough_share'] = go['_pass'].mean()
    F['h_after_in_median'] = go['h_after_in'].median()
    F['amt_diff_after_in_median'] = go['amt_diff'].median()
    del out; gc.collect()

    # 3.3 Repeated amounts
    tx['_ar'] = tx['amt'].round(2)
    rep = tx.groupby(['signal_id', '_ar']).size()
    rg = rep.groupby(level=0)
    cnt = tx.groupby('signal_id').size()
    F['amt_unique_ratio'] = rg.size() / cnt
    F['amt_top_repeat_share'] = rg.max() / cnt
    F['amt_repeated_cnt'] = rep[rep >= 3].groupby(level=0).sum()

    # 3.4 Last 10 transactions
    tx['_rk'] = tx.groupby('signal_id').cumcount(ascending=False)
    last = tx[tx['_rk'] < 10].groupby('signal_id')
    F['last10_amt_mean'] = last['amt'].mean()
    F['last10_amt_max'] = last['amt'].max()
    F['last10_in_share'] = last['is_in'].mean()
    F['last10_naqd_share'] = last['is_naqd'].mean()
    F['last10_xalq_share'] = last['is_xalq'].mean()
    F['last10_span_days'] = last['dt'].max() - last['dt'].min()

    # 3.5 Type entropy
    p = tx.groupby(['signal_id', 'tranzaksiya_turi']).size().unstack(fill_value=0)
    p = p.div(p.sum(axis=1), axis=0)
    F['type_entropy'] = -(p * np.log(p + 1e-9)).sum(axis=1)
    F['n_types'] = (p > 0).sum(axis=1)

    # 3.6 Unusual recent amounts vs the alert's own history
    stats = tx.groupby('signal_id')['amt'].agg(['mean', 'std'])
    r7 = tx[tx['dt'].between(0, 7)].groupby('signal_id')['amt'].agg(['max', 'mean'])
    F['z_max7'] = (r7['max'] - stats['mean']) / (stats['std'] + 1e-6)
    F['z_mean7'] = (r7['mean'] - stats['mean']) / (stats['std'] + 1e-6)
    old = tx[tx['dt'] > 30].groupby('signal_id')['amt'].mean()
    new = tx[tx['dt'].between(0, 30)].groupby('signal_id')['amt'].mean()
    F['amt_mean_30_vs_older'] = new - old

    fill0 = [c for c in F.columns if c.endswith('_cnt')]
    F[fill0] = F[fill0].fillna(0)
    return F.astype('float32')

# %%
def make_all(sig, tx, name):
    path = f'{CACHE}/features_{name}.parquet'
    if USE_CACHE and os.path.exists(path):
        print('Loaded from cache:', path)
        return pd.read_parquet(path)
    if DROP_AFTER_SIGNAL:
        t = tx.merge(sig[['signal_id', 'signal_sanasi']], on='signal_id')
        tx = tx[(pd.to_datetime(t['tranzaksiya_vaqti']) <= signal_end(t['signal_sanasi'])).values]
    t0 = time.time()
    X = build_features(sig, tx).join(build_extra(sig, tx))
    X.to_parquet(path)
    print(f'{name}: {X.shape}, {time.time() - t0:.0f} s')
    return X

X_tr = make_all(tr_sig, tr_tx, 'train')
X_te = make_all(te_sig, te_tx, 'test').reindex(columns=X_tr.columns)
y = tr_sig.set_index('signal_id').loc[X_tr.index, 'eskalatsiya'].values
del tr_tx, te_tx; gc.collect()
print('Number of features:', X_tr.shape[1])

# %% [markdown]
# ## 4. Adversarial validation
# We train a model to distinguish train rows from test rows.
# 
# **How to read Adversarial AUC:**
# - **about 0.5** — train and test are indistinguishable. Good.
# - **above 0.7** — there is a distribution shift. The notebook then drops the most shifting features one by one (up to 10), because they may hurt on the hidden test.

# %%
def adversarial(X_a, X_b, rounds=300):
    X = pd.concat([X_a, X_b]); t = np.r_[np.zeros(len(X_a)), np.ones(len(X_b))]
    oof = np.zeros(len(X)); imp = pd.Series(0.0, index=X.columns)
    p = dict(objective='binary', learning_rate=0.05, num_leaves=31, feature_fraction=0.8, verbose=-1, seed=SEED)
    for i_tr, i_va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, t):
        m = lgb.train(p, lgb.Dataset(X.iloc[i_tr], t[i_tr]), rounds)
        oof[i_va] = m.predict(X.iloc[i_va])
        imp += pd.Series(m.feature_importance('gain'), index=X.columns)
    return roc_auc_score(t, oof), imp.sort_values(ascending=False)

adv_auc, adv_imp = adversarial(X_tr, X_te)
print(f'Adversarial AUC: {adv_auc:.4f}')

DROP = []
feats = list(X_tr.columns)
while adv_auc > ADV_DROP_THRESHOLD and len(DROP) < 10:
    top = adv_imp.index[0]
    DROP.append(top); feats.remove(top)
    adv_auc, adv_imp = adversarial(X_tr[feats], X_te[feats])
    print(f'  dropped {top:35s} -> adversarial AUC {adv_auc:.4f}')
print('Dropped features:', DROP or 'none')

# %% [markdown]
# ## 4b. Automatic tuning: feature count and LightGBM settings
# The target signal is weak (AUC around 0.64), so noisy features and overly flexible models easily overfit. This section runs quick experiments and **keeps whichever setup gives the highest cross-validated AUC**:
# 
# 1. **Feature selection:** rank features by LightGBM importance, then compare keeping the top 30 / 50 / 80 / 120 / all.
# 2. **LightGBM settings:** compare the default setup with more regularised ones (fewer leaves, more samples per leaf, fewer features per tree).
# 
# Experiments use a faster learning rate; the final models in section 5 are retrained with the chosen setup. Set `RUN_SEARCH = False` in section 0 to skip this step.

# %%
LGB_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.02, num_leaves=31,
                  min_child_samples=50, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l1=0.1, lambda_l2=2.0, verbose=-1)
CANDIDATES = {
    'default':      {},
    'regularised':  dict(num_leaves=15, min_child_samples=100, feature_fraction=0.5, lambda_l2=5.0),
    'strong_reg':   dict(num_leaves=7, min_child_samples=200, feature_fraction=0.4, lambda_l2=10.0),
    'shallow_wide': dict(num_leaves=31, max_depth=4, min_child_samples=150, feature_fraction=0.3),
}
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_tr, y))

def quick_cv(cols, extra=None):
    p = {**LGB_PARAMS, **(extra or {}), 'learning_rate': 0.05, 'seed': SEED}
    oof = np.zeros(len(X_tr)); imp = pd.Series(0.0, index=cols)
    for i_tr, i_va in folds:
        m = lgb.train(p, lgb.Dataset(X_tr.iloc[i_tr][cols], y[i_tr]), 3000,
                      valid_sets=[lgb.Dataset(X_tr.iloc[i_va][cols], y[i_va])],
                      callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[i_va] = m.predict(X_tr.iloc[i_va][cols], num_iteration=m.best_iteration)
        imp += m.feature_importance('gain')
    return roc_auc_score(y, oof), imp.sort_values(ascending=False)

SEARCH_LOG = []
if RUN_SEARCH:
    base_auc, ranking = quick_cv(feats)
    print(f'All {len(feats)} features, default settings: AUC {base_auc:.4f}')
    best_auc, best_k = base_auc, None
    for k in [k for k in TOP_K_GRID if k and k < len(feats)]:
        a, _ = quick_cv(list(ranking.index[:k]))
        SEARCH_LOG.append({'step': 'features', 'top_k': k, 'auc': round(a, 4)})
        print(f'  top {k:3d} features: AUC {a:.4f}')
        if a > best_auc + 0.0005: best_auc, best_k = a, k
    if best_k:
        feats = list(ranking.index[:best_k])
    print(f'-> keeping {len(feats)} features')

    best_name = 'default'
    for name, extra in CANDIDATES.items():
        if name == 'default': continue
        a, _ = quick_cv(feats, extra)
        SEARCH_LOG.append({'step': 'params', 'config': name, 'auc': round(a, 4)})
        print(f'  settings {name:13s}: AUC {a:.4f}')
        if a > best_auc + 0.0005: best_auc, best_name = a, name
    LGB_PARAMS.update(CANDIDATES[best_name])
    print(f'-> chosen LightGBM settings: {best_name} (quick-CV AUC {best_auc:.4f})')

# %% [markdown]
# ## 5. Models and cross-validation
# Three gradient-boosting models: LightGBM, CatBoost and XGBoost.
# 
# **5-fold cross-validation:** the training set is split into 5 parts; each model is trained on 4 parts and evaluated on the 5th, five times. Every alert thus gets an out-of-fold (OOF) prediction from a model that never saw it, and **OOF AUC** is an honest quality estimate. Test predictions are averaged over the 5 fold models.
# 
# **Reading AUC:** 0.5 = random guessing, 1.0 = perfect ranking.

# %%
Xtr, Xte = X_tr[feats], X_te[feats]
OOF, PRED, IMP = {}, {}, pd.Series(0.0, index=feats)

def run_lgb(seed):
    oof, pred = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for i_tr, i_va in folds:
        m = lgb.train({**LGB_PARAMS, 'seed': seed}, lgb.Dataset(Xtr.iloc[i_tr], y[i_tr]), 10000,
                      valid_sets=[lgb.Dataset(Xtr.iloc[i_va], y[i_va])],
                      callbacks=[lgb.early_stopping(300, verbose=False)])
        oof[i_va] = m.predict(Xtr.iloc[i_va], num_iteration=m.best_iteration)
        pred += m.predict(Xte, num_iteration=m.best_iteration) / N_FOLDS
        IMP.loc[:] += m.feature_importance('gain')
    return oof, pred

res = [run_lgb(s) for s in LGB_SEEDS]
OOF['lgb'] = np.mean([r[0] for r in res], axis=0)
PRED['lgb'] = np.mean([r[1] for r in res], axis=0)
print(f'Features used: {len(feats)}')
print(f"LightGBM ({len(LGB_SEEDS)} seeds) OOF AUC: {roc_auc_score(y, OOF['lgb']):.4f}")

# %%
oof, pred = np.zeros(len(Xtr)), np.zeros(len(Xte))
for i_tr, i_va in folds:
    m = CatBoostClassifier(iterations=5000, learning_rate=0.03, depth=6, l2_leaf_reg=5,
                           eval_metric='AUC', random_seed=SEED, od_type='Iter', od_wait=300,
                           verbose=0, thread_count=-1, allow_writing_files=False)
    m.fit(Xtr.iloc[i_tr], y[i_tr], eval_set=(Xtr.iloc[i_va], y[i_va]), use_best_model=True)
    oof[i_va] = m.predict_proba(Xtr.iloc[i_va])[:, 1]
    pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
OOF['cat'], PRED['cat'] = oof, pred
print(f"CatBoost OOF AUC: {roc_auc_score(y, OOF['cat']):.4f}")

# %%
XGB_PARAMS = dict(objective='binary:logistic', eval_metric='auc', eta=0.02, max_depth=6,
                  min_child_weight=5, subsample=0.8, colsample_bytree=0.6, reg_lambda=2.0,
                  tree_method='hist', seed=SEED, nthread=-1)
dte = xgb.DMatrix(Xte)
oof, pred = np.zeros(len(Xtr)), np.zeros(len(Xte))
for i_tr, i_va in folds:
    dtr = xgb.DMatrix(Xtr.iloc[i_tr], y[i_tr]); dva = xgb.DMatrix(Xtr.iloc[i_va], y[i_va])
    m = xgb.train(XGB_PARAMS, dtr, 10000, evals=[(dva, 'va')], early_stopping_rounds=300, verbose_eval=False)
    rng = (0, m.best_iteration + 1)
    oof[i_va] = m.predict(dva, iteration_range=rng)
    pred += m.predict(dte, iteration_range=rng) / N_FOLDS
OOF['xgb'], PRED['xgb'] = oof, pred
print(f"XGBoost OOF AUC: {roc_auc_score(y, OOF['xgb']):.4f}")

# %% [markdown]
# ## 6. Time-based holdout check
# Train on the earliest 80% of alerts and evaluate on the latest 20%, imitating "learn from the past, predict the future".
# 
# Compare **Time holdout AUC** with **OOF AUC**: similar values mean the model is stable over time; a much lower holdout means it memorises period-specific patterns.

# %%
dates = pd.to_datetime(tr_sig.set_index('signal_id').loc[Xtr.index, 'signal_sanasi'])
early = (dates <= dates.quantile(0.8)).values
m = lgb.train({**LGB_PARAMS, 'seed': SEED}, lgb.Dataset(Xtr[early], y[early]), 10000,
              valid_sets=[lgb.Dataset(Xtr[~early], y[~early])],
              callbacks=[lgb.early_stopping(300, verbose=False)])
time_auc = roc_auc_score(y[~early], m.predict(Xtr[~early], num_iteration=m.best_iteration))
print(f'Time holdout AUC (LightGBM): {time_auc:.4f}  |  OOF AUC: {roc_auc_score(y, OOF["lgb"]):.4f}')

# %% [markdown]
# ## 7. Ensemble
# ROC-AUC depends only on the **ordering** of predictions, so each model's predictions are converted to ranks and blended with weights chosen on OOF predictions (coarse grid). If no blend beats the best single model, the best single model is used.

# %%
def rank(p): return rankdata(p) / len(p)
names = list(OOF)
scores = {n: roc_auc_score(y, OOF[n]) for n in names}
best = (max(scores.values()), None)
grid = np.arange(0, 1.01, 0.1)
for w1 in grid:
    for w2 in grid:
        if w1 + w2 > 1: continue
        w = dict(zip(names, [w1, w2, 1 - w1 - w2]))
        s = roc_auc_score(y, sum(w[n] * rank(OOF[n]) for n in names))
        if s > best[0] + 1e-5: best = (s, w)

for n in names: print(f'{n:5s} OOF AUC {scores[n]:.4f}')
if best[1] is None:
    top = max(scores, key=scores.get)
    final_pred, final_auc, weights = PRED[top], scores[top], {top: 1.0}
else:
    weights = {k: round(v, 2) for k, v in best[1].items()}
    final_pred = sum(best[1][n] * rank(PRED[n]) for n in names)
    final_auc = best[0]
print('Ensemble weights:', weights, f'| Ensemble OOF AUC: {final_auc:.4f}')

# %% [markdown]
# ## 8. Feature importance and files for the website
# Saves two files into `app/assets/`, which the website (`app/app.py`) reads automatically:
# - `feature_importance.csv` — importance of every feature (share of total LightGBM gain);
# - `metrics.json` — all final metrics.

# %%
imp = (IMP / IMP.sum()).sort_values(ascending=False)
imp.rename('importance').rename_axis('feature').reset_index().to_csv(f'{ASSETS}/feature_importance.csv', index=False)
metrics = {'oof_auc': round(float(final_auc), 4), 'time_holdout_auc': round(float(time_auc), 4),
           'model_auc': {n: round(float(s), 4) for n, s in scores.items()}, 'weights': weights,
           'adversarial_auc': round(float(adv_auc), 4), 'dropped_features': DROP,
           'n_features': len(feats), 'n_train': int(len(Xtr)), 'n_test': int(len(Xte)),
           'target_rate': round(float(y.mean()), 4),
           'lgb_params': {k: v for k, v in LGB_PARAMS.items() if k not in ('verbose', 'metric', 'objective')},
           'search_log': SEARCH_LOG, **DATA_STATS}
json.dump(metrics, open(f'{ASSETS}/metrics.json', 'w'), indent=2)
print(json.dumps(metrics, indent=2))
print()
print('Top-20 features:')
print(imp.head(20).round(4).to_string())

# %% [markdown]
# ## 9. Submission
# Saves `submission/team_E86FDEB5.csv` and checks every competition rule: one row per test alert, no duplicates or missing values, probabilities in [0, 1], exactly two columns `signal_id,ehtimollik`, no index column.

# %%
sub = pd.DataFrame({'signal_id': Xte.index, 'ehtimollik': np.clip(final_pred, 0, 1)})
sub = sub.set_index('signal_id').loc[te_sig['signal_id']].reset_index()

assert len(sub) == len(te_sig), 'row count mismatch'
assert sub['signal_id'].is_unique, 'duplicate IDs'
assert set(sub['signal_id']) == set(sample['signal_id']), 'IDs do not match sample_submission'
assert sub['ehtimollik'].notna().all(), 'missing predictions'
assert sub['ehtimollik'].between(0, 1).all(), 'values outside [0, 1]'
assert list(sub.columns) == ['signal_id', 'ehtimollik']

fname = f'{SUBMIT}/team_{TEAM_ID}.csv'
sub.to_csv(fname, index=False)
chk = pd.read_csv(fname)
assert list(chk.columns) == ['signal_id', 'ehtimollik'] and len(chk) == len(te_sig)
print('Saved and validated:', fname, chk.shape)
print(f'Total runtime: {(time.time() - T0) / 60:.1f} min')

train signals (14000, 3) | test signals (6000, 2)
train tx (6987663, 5) | test tx (3027575, 5)
Escalation rate: 0.1718
Signal dates train: 2025-01-01 — 2026-12-31
Signal dates test: 2025-01-01 — 2026-12-31
{'n_tx_train': 6987663, 'n_tx_test': 3027575, 'tx_per_alert_median': 461.0, 'train_dates': ['2025-01-01', '2026-12-31'], 'test_dates': ['2025-01-01', '2026-12-31']}
Share of transactions after the signal date: train 0.0 | test 0.0
Loaded from cache: ../data/cache/features_train.parquet
Loaded from cache: ../data/cache/features_test.parquet
Number of features: 165
Adversarial AUC: 0.4987
Dropped features: none
All 165 features, default settings: AUC 0.6268
  top  30 features: AUC 0.6327
  top  50 features: AUC 0.6330
  top  80 features: AUC 0.6346
  top 120 features: AUC 0.6324
-> keeping 80 features
  settings regularised  : AUC 0.6362
  settings strong_reg   : AUC 0.6395
  settings shallow_wide : AUC 0.6373
-> chosen LightGBM settings: strong_reg (quick-CV AUC 0.6395)
Features used: